# Phase 4 — the evidence pathway (Stage C)

Trains **M2**, the half of the project that looks for lesions rather than grading the
image. Its independence from M1 is the thesis: if the two shared an encoder they would
fail together and disagreement between them would carry no information.

| | | |
|---|---|---|
| **C1** | Optic disc / fovea regressor (M2b) | Gate: mean error **< 0.5 disc diameters** |
| **C2** | Four-channel lesion segmenter (M2a) | Per-lesion Dice and IoU |
| **C3** | Cross-domain: DDR ↔ IDRiD | Does Dice survive a change of source? |

C1 comes first because everything else rests on it. Without disc and fovea positions
there is no coordinate frame, and "haemorrhages in three quadrants" — the 4-2-1 rule
M3 reasons with — is undefined.

## What can block each one

They are **independent**, and section 5 reports them separately.

- **C1** needs IDRiD **Part C** centre coordinates. IDRiD ships Part A (81 images,
  lesion masks, `IDRiD_01`–`81`) and Part B/C (516 images, grades and coordinates,
  `IDRiD_001`–`516`) as *different image sets*. A cache holding only Part A gives the
  coordinates nothing to join to.
- **C2** needs lesion masks on disk. IDRiD's ship in a separate published dataset from
  the main cache, so attach **both**.

A blocked C1 does not stop C2 — the segmenter simply starts from ImageNet weights
rather than C1's encoder, which is a recorded deviation, not a blocker.

## Notebook settings

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Persistence | **Files only** |
| Internet | **On** (git clone) |
| Environment | **Pin to original environment** |

Inputs, all three: **`verify-dr-cache-512`**, **`verify-dr-manifests`**, and the IDRiD
cache — **`verify-dr-idrid`** once `01c_idrid_grading.ipynb` has been run and published,
otherwise `verify-dr-idrid-masks` (which carries Part A's masks but not the Part B images
C1 needs).

## Cost

C1 is ~10–15 GPU-minutes. C2 on roughly 840 annotated images is ~45–60. C3 is two more
runs at ~30 each. Budget about **2 GPU-hours** for the whole notebook; set `RUN_C3 =
False` in section 8 to stop after C2.

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Accelerator to 'GPU T4 x2', then re-run from the top.")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB")

## 3 · Helpers

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a training job, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"training failed with exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

## 4 · Find the cache and the manifests

In [ ]:
ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()

if not ROOTS:
    raise RuntimeError(
        "No cache found. Attach verify-dr-cache-512 (and verify-dr-idrid-masks) "
        "under Add Data in the right-hand panel.")
if MANIFEST_DIR is None:
    raise RuntimeError(
        "No manifests found. Attach the Phase 2 output (verify-dr-manifests), or "
        "add 02_manifests.ipynb as a notebook input.")

DATASETS = resolve_datasets(ROOTS)
print("cache roots  :")
for r in ROOTS:
    print("   ", r)
print("manifest dir :", MANIFEST_DIR)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print("\ndatasets in the cache:")
print(f"  {'dataset':<12}{'images':>8}  {'masks':>5}  root")
for name, root in sorted(DATASETS.items()):
    d = root / name
    images = d / 'images'
    # rglob, not glob: glob('*') counts direct children only, so a nested
    # layout reports 1 and looks like catastrophic data loss when nothing
    # is actually wrong.
    n_img = sum(1 for f in images.rglob('*') if f.suffix.lower() in IMAGE_SUFFIXES) \
        if images.is_dir() else 0
    n_msk = len([m for m in (d / 'masks').iterdir() if m.is_dir()]) \
        if (d / 'masks').is_dir() else 0
    print(f"  {name:<12}{n_img:>8}  {n_msk:>5}  {root}")

    if images.is_dir():
        subdirs = [x for x in images.glob('*') if x.is_dir()]
        if subdirs and n_img:
            print(f"               ^ nested under {len(subdirs)} subdirectorie(s), "
                  f"e.g. {subdirs[0].name}/ - fine, the manifest stores full paths")
    if n_img == 0:
        print(f"               ^ NO IMAGES - {name} is empty in every attached root")

print("\nmanifests available:")
for c in sorted(MANIFEST_DIR.glob("*.csv")):
    print("  ", c.name)

# Phase 3 could ignore this column. Phase 4 cannot: C2 IS the mask experiment.
mask_bearing = [n for n, r in sorted(DATASETS.items())
                if (r / n / "masks").is_dir()
                and any(x.is_dir() for x in (r / n / "masks").iterdir())]
print()
if mask_bearing:
    print(f"mask channels present in: {mask_bearing}")
else:
    print("NO dataset in any attached root has mask channels.")
    print("C2 and C3 cannot run. Attach verify-dr-idrid-masks (or verify-dr-idrid)")
    print("alongside verify-dr-cache-512 -- IDRiD's masks ship separately from the")
    print("main cache. Section 5 below re-checks this per experiment.")

# Every root goes to --cache-root, so each dataset resolves to the root that
# actually holds it instead of all of them being forced onto one.
CACHE_FLAGS = ' '.join(q(r) for r in ROOTS)

## 5 · What can actually run?

Checked before any GPU time is spent. Each experiment is reported on its own, with the
cause and the fix when it is blocked.

In [ ]:
import sys
import pandas as pd
from pathlib import Path as _P

# The same functions the training scripts use, so this precondition tests what
# the run will actually see. Manifests store the paths the cache had in the
# Phase 2 session; repath_to_cache re-points them at this session's mounts, and
# reimplementing it here would let the check disagree with the trainer.
sys.path.insert(0, str(REPO_DIR / "src"))
from verify_dr.data.dataset import repath_to_cache
from verify_dr.data.segmentation import mask_path

GEOM_COLS = ['od_x', 'od_y', 'fovea_x', 'fovea_y']
MASK_DIRS = ['MA', 'HE', 'EX', 'SE']

MANIFESTS = sorted(MANIFEST_DIR.glob('*.csv'))
print('manifests available:', [m.name for m in MANIFESTS])

# ---- C1 precondition: IDRiD Part C centres --------------------------------
C1_MANIFEST, C1_READY, C1_WHY = None, False, ''
for name in ('idrid_manifest.csv', 'idrid.csv'):
    if (MANIFEST_DIR / name).exists():
        C1_MANIFEST = MANIFEST_DIR / name
        break

if C1_MANIFEST is None:
    C1_WHY = 'no IDRiD manifest found'
else:
    f = pd.read_csv(C1_MANIFEST)
    missing = [c for c in GEOM_COLS if c not in f.columns]
    if missing:
        C1_WHY = ('manifest has no coordinate columns. prepare_manifest.py ran '
                  'without --coords, or the Part C tables were not found.')
    else:
        usable = f.dropna(subset=GEOM_COLS)
        for flag in ('od_in_frame', 'fovea_in_frame'):
            if flag in usable.columns:
                usable = usable[usable[flag].astype(int) == 1]
        if len(usable) == 0:
            stems = sorted({_P(x).stem for x in f['image_path']})
            C1_WHY = ('no cached image carries both centres. This is the Part A / '
                      'Part C split: Part A is 81 images with masks (IDRiD_01-81), '
                      'Part C covers the 516 grading images (IDRiD_001-516). '
                      f'The cache holds {len(f)} images named like {stems[:2]}. '
                      'Fix: cache IDRiD Part B images in Phase 1, then re-run Phase 2.')
        else:
            C1_READY = True
            C1_WHY = f'{len(usable)} of {len(f)} images have both centres'

# ---- C2 precondition: lesion masks on disk --------------------------------
C2_MANIFESTS, mask_counts = [], {}
for m in MANIFESTS:
    f = pd.read_csv(m)
    if 'dataset' not in f.columns or 'image_path' not in f.columns:
        continue
    if not set(f['dataset'].str.lower()) & {'ddr', 'idrid'}:
        continue
    sub = f[f['dataset'].str.lower().isin(['ddr', 'idrid'])].copy()
    try:
        sub = repath_to_cache(sub, ROOTS)
    except Exception as exc:
        print(f'  {m.name}: could not repath to this session\'s cache ({exc})')
        continue
    found = 0
    for path in sub['image_path'].head(400):
        if any(mask_path(path, ch).exists() for ch in MASK_DIRS):
            found += 1
    if found:
        C2_MANIFESTS.append(m)
        mask_counts[m.name] = found

C2_READY = bool(C2_MANIFESTS)

print()
print('=' * 72)
print(f"C1  (disc/fovea)    {'READY' if C1_READY else 'BLOCKED'}   {C1_WHY}")
if C2_READY:
    print(f'C2  (lesion masks)  READY   masks found in {mask_counts}')
else:
    print('C2  (lesion masks)  BLOCKED  no manifest row has a mask on disk,')
    print('                    after repathing onto this session\'s cache roots.')
    print('                    Attach verify-dr-idrid (or verify-dr-idrid-masks)')
    print('                    alongside verify-dr-cache-512.')
print('=' * 72)
if not (C1_READY or C2_READY):
    raise RuntimeError('Neither C1 nor C2 can run. Fix the causes above first.')
print()
print('The two are independent: a blocked C1 does not stop C2. C2 simply starts')
print('from ImageNet weights instead of C1-s encoder, which is a recorded')
print('deviation rather than a blocker.')

## 6 · C1 — optic disc and fovea

Error is reported in **disc diameters**, not pixels: a fixed pixel tolerance means
different things at different fields of view. IDRiD publishes no disc diameter, so it
is derived per image from the disc-to-fovea distance ÷ 2.5, the standard clinical
relation.

Watch the **constant-predictor baseline** as closely as the gate. Fundus framing is
stereotyped, so predicting the training mean scores better than intuition suggests. A
model that barely beats it has learned the average layout, not this image's landmarks —
and the gate would then be passing for the wrong reason.

In [ ]:
C1_EXPERIMENT = 'C1_geometry'
C1_BEST = RESULTS / C1_EXPERIMENT / 'best.pt'

if not C1_READY:
    print('SKIPPED -', C1_WHY)
else:
    run(' '.join([
        f"python {q(REPO_DIR / 'scripts/train_geometry.py')}",
        f'--manifest {q(C1_MANIFEST)}',
        f'--experiment {q(C1_EXPERIMENT)}',
        f'--results-dir {q(RESULTS)}',
        f'--cache-root {CACHE_FLAGS}',
        '--image-size 512 --batch-size 16 --epochs 40 --lr 3e-4',
        '--val-frac 0.2 --patience 10 --workers 2 --resume',
    ]))

## 7 · C2 — the four-channel lesion segmenter

Microaneurysm, haemorrhage, hard exudate, soft exudate. Only four, because these are
the only lesion types with pixel annotations on a Kaggle-reachable dataset — venous
beading, IRMA and neovascularisation are not labelled anywhere available, and M3
declares them unobservable rather than pretending.

`L = 0.5·Dice + 0.5·BCE`. Dice alone is unstable when a channel is empty, and empty
channels are the norm rather than the exception: most fundus images carry no soft
exudates at all.

**Read Dice over images where the lesion is annotated**, which is what the table
reports. Averaging over every image folds in a long run of empty-target/empty-prediction
pairs that score 1.0 by convention, inflating the headline without the model having
segmented anything.

In [ ]:
C2_EXPERIMENT = 'C2_lesions'

if not C2_READY:
    print('SKIPPED - no lesion masks found on disk.')
else:
    flags = [
        f"python {q(REPO_DIR / 'scripts/train_evidence.py')}",
        '--manifest ' + ' '.join(q(m) for m in C2_MANIFESTS),
        f'--experiment {q(C2_EXPERIMENT)}',
        f'--results-dir {q(RESULTS)}',
        f'--cache-root {CACHE_FLAGS}',
        '--image-size 512 --batch-size 4 --epochs 60 --lr 3e-4',
        '--val-frac 0.2 --patience 15 --workers 2 --resume',
    ]
    # docs/03 has M2a and M2b sharing an encoder. Their supervision is disjoint,
    # so they are fitted in sequence: geometry first, then the decoder on top.
    if C1_BEST.exists():
        flags.append(f'--encoder-from {q(C1_BEST)}')
    else:
        print('note: no C1 checkpoint, so the encoder starts from ImageNet.')
        print('      Record that as a deviation from the shared-encoder design.')
    run(' '.join(flags))

## 8 · C3 — does it survive a change of source?

Train on one dataset, test on the other, both directions. This is the experiment that
says whether the evidence pathway means anything off-domain.

If Dice collapses here, disagreement measured on one source says little about another,
and the thesis has to state that plainly. Set `RUN_C3 = False` to stop after C2.

In [ ]:
# C3: does the segmenter survive a change of source? If Dice collapses, the
# disagreement signal is noise off-domain, and the thesis has to say so.
RUN_C3 = True

if not (C2_READY and RUN_C3):
    print('SKIPPED')
else:
    import pandas as pd
    sources = set()
    for m in C2_MANIFESTS:
        sources |= set(pd.read_csv(m)['dataset'].str.lower())
    sources &= {'ddr', 'idrid'}

    if len(sources) < 2:
        print(f'SKIPPED - cross-domain needs both DDR and IDRiD, found {sources}.')
    else:
        for train_on, test_on in (('ddr', 'idrid'), ('idrid', 'ddr')):
            print()
            print('=' * 72)
            print(f'C3: train {train_on} -> test {test_on}')
            print('=' * 72, flush=True)
            run(' '.join([
                f"python {q(REPO_DIR / 'scripts/train_evidence.py')}",
                '--manifest ' + ' '.join(q(m) for m in C2_MANIFESTS),
                f'--experiment C3_{train_on}_to_{test_on}',
                f'--results-dir {q(RESULTS)}',
                f'--cache-root {CACHE_FLAGS}',
                f'--train-datasets {train_on} --val-datasets {test_on}',
                '--image-size 512 --batch-size 4 --epochs 40 --lr 3e-4',
                '--patience 12 --workers 2 --resume',
            ]))

## 9 · Stage C summary

In [ ]:
import pandas as pd

def load(name):
    path = RESULTS / name / 'metrics.json'
    return json.loads(path.read_text()) if path.exists() else None

print('=' * 72)
print('PHASE 4 — STAGE C')
print('=' * 72)

c1 = load('C1_geometry')
if c1 is None:
    print('\nC1  not run.')
else:
    b, base = c1['best_val'], c1['baseline']
    lift = base['mean_error_dd'] - c1['best_mean_error_dd']
    print(f"\nC1  optic disc / fovea     {c1['minutes']:.0f} min")
    print(f"    mean error        {c1['best_mean_error_dd']:.3f} DD   gate 0.5")
    print(f"    optic disc        {b['od_error_px']:.1f} px  ({b['od_error_dd']:.3f} DD)")
    print(f"    fovea             {b['fovea_error_px']:.1f} px  ({b['fovea_error_dd']:.3f} DD)")
    print(f"    within 0.5 DD     {b['within_half_dd'] * 100:.1f}%")
    print(f"    vs constant       {lift:+.3f} DD")
    verdict = 'PASS' if c1['best_mean_error_dd'] < 0.5 else 'FAIL'
    print(f"    -> C1 {verdict}" + ('' if verdict == 'PASS' else '  (M3 falls back to count-only)'))
    if lift <= 0:
        print('    WARNING: no better than the training mean. It learned the average layout.')

c2 = load('C2_lesions')
if c2 is None:
    print('\nC2  not run.')
else:
    print(f"\nC2  lesion segmentation    {c2['minutes']:.0f} min   "
          f"{c2['train_images']} train / {c2['val_images']} val")
    print(f"    mean Dice (present) {c2['best_mean_dice_present']:.4f}")
    rows = []
    for name, v in c2['best_val']['per_lesion'].items():
        rows.append({'lesion': name, 'Dice': v['dice_present'], 'IoU': v['iou_present'],
                     'images': v['images_with_lesion'], 'FP imgs': v['false_positive_images'],
                     'pred px': round(v['mean_pred_px']), 'true px': round(v['mean_truth_px'])})
    print(pd.DataFrame(rows).to_string(index=False))
    print('    Dice is over images where the lesion is annotated, not all images.')

print()
cross = {k: load(k) for k in ('C3_ddr_to_idrid', 'C3_idrid_to_ddr')}
if any(cross.values()):
    print('C3  cross-domain transfer')
    base = c2['best_mean_dice_present'] if c2 else None
    for name, r in cross.items():
        if r is None:
            continue
        d = r['best_mean_dice_present']
        gap = f"  ({d - base:+.4f} vs C2)" if base else ''
        print(f"    {name:<20} mean Dice {d:.4f}{gap}")
    vals = [r['best_mean_dice_present'] for r in cross.values() if r]
    if base and vals and min(vals) < 0.5 * base:
        print()
        print('    Dice more than halves off-domain. The evidence pathway does not')
        print('    transfer, so disagreement measured on one source says little about')
        print('    another. That is a finding for the thesis, not a bug to hide.')
else:
    print('C3  not run.')
print()
print('=' * 72)

---
## 10 · Save

**Save Version → Save & Run All (Commit).** Publish `/kaggle/working/results` as
**`verify-dr-stage-c`**. Phase 6 loads these checkpoints, so they have to outlive the
session.

Record C1–C3 in `docs/04_experiment_register.md`: C1's mean error in disc diameters
with the baseline margin and whether the gate passed, C2's per-lesion Dice with the
image counts beside them, and C3's two transfer figures against C2's in-domain number.

> **C4 — evidence-only grading — is not in this notebook.** It needs the reasoner
> (M3), which Phase 4 does not build. `docs/00_START_HERE.md` calls C4 load-bearing:
> the evidence path has to be *informative but weaker* than the grader. If it matches
> M1 you do not need M1; if it is noise, disagreement means nothing.